# Infant Cry Classification: MFCC vs Pain Markers vs Combined
This notebook extracts baseline MFCC features and pain-specific acoustic markers, trains RandomForest models, and compares performance with robust evaluation and explainability.

**Goals**
- Research-grade, modular feature pipelines
- Model comparison (MFCC, pain markers, combined)
- Feature importance and SHAP explainability
- Clear evaluation metrics and confusion matrices

## Setup and Data
This notebook expects raw audio under `mlModels/CryTranslater/data/raw/Audio/Baby Cry Dataset/` organized by class folders.

If your audio format requires ffmpeg support, ensure it is installed and accessible (librosa uses `audioread`).

In [30]:
# Optional: uncomment if packages are missing

# %pip install librosa parselmouth shap seaborn scikit-learn pandas numpy

In [31]:
import os
import warnings
from dataclasses import dataclass, field
from typing import List, Tuple, Dict
import numpy as np
import pandas as pd
import librosa
import parselmouth
from parselmouth import praat
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
    )
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import SelectFromModel
try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print("XGBoost not installed. Install with: pip install xgboost")

warnings.filterwarnings("ignore")
sns.set(style="whitegrid")
np.random.seed(42)

XGBoost not installed. Install with: pip install xgboost


In [32]:
@dataclass
class Config:
    data_root: str = os.path.join("..", "data", "raw", "Audio", "Baby Cry Dataset")
    cache_dir: str = os.path.join("..", "data", "processed", "feature_cache")
    sample_rate: int = 22050
    n_mfcc: int = 20
    n_mels: int = 64
    hop_length: int = 512
    frame_length: int = 2048
    test_size: float = 0.2
    random_state: int = 42
    target_classes: Tuple[str, ...] = ("pain", "no_pain", "hungry")
    label_version: str = "pain_no_pain_hungry"
    label_map: Dict[str, str] = field(default_factory=lambda: {
        "belly_pain": "pain",
        "discomfort": "pain",
        "cold_hot": "pain",
        "hungry": "hungry",
        "burping": "no_pain",
        "lonely": "no_pain",
        "scared": "no_pain",
        "tired": "no_pain",
    })
    pain_class_names: Tuple[str, ...] = ("pain",)

cfg = Config()
os.makedirs(cfg.cache_dir, exist_ok=True)
cfg

Config(data_root='..\\data\\raw\\Audio\\Baby Cry Dataset', cache_dir='..\\data\\processed\\feature_cache', sample_rate=22050, n_mfcc=20, n_mels=64, hop_length=512, frame_length=2048, test_size=0.2, random_state=42, target_classes=('pain', 'no_pain', 'hungry'), label_version='pain_no_pain_hungry', label_map={'belly_pain': 'pain', 'discomfort': 'pain', 'cold_hot': 'pain', 'hungry': 'hungry', 'burping': 'no_pain', 'lonely': 'no_pain', 'scared': 'no_pain', 'tired': 'no_pain'}, pain_class_names=('pain',))

In [33]:
def _safe_float(value: float) -> float:
    if value is None or np.isnan(value) or np.isinf(value):
        return 0.0
    return float(value)

def load_audio(file_path: str, sr: int) -> Tuple[np.ndarray, int]:
    y, sr = librosa.load(file_path, sr=sr, mono=True)
    if y.size == 0:
        raise ValueError("Empty audio")
    return y, sr

def extract_mfcc_features(y: np.ndarray, sr: int, cfg: Config) -> Tuple[np.ndarray, List[str]]:
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=cfg.n_mfcc)
    mfcc_mean = mfcc.mean(axis=1)
    mfcc_std = mfcc.std(axis=1)

    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=cfg.n_mels)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_mean = mel_db.mean(axis=1)

    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_mean = chroma.mean(axis=1)

    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    contrast_mean = contrast.mean(axis=1)

    features = np.concatenate([mfcc_mean, mfcc_std, mel_mean, chroma_mean, contrast_mean])
    names = (
        [f"mfcc_mean_{i}" for i in range(cfg.n_mfcc)] +
        [f"mfcc_std_{i}" for i in range(cfg.n_mfcc)] +
        [f"mel_mean_{i}" for i in range(cfg.n_mels)] +
        [f"chroma_mean_{i}" for i in range(chroma_mean.shape[0])] +
        [f"contrast_mean_{i}" for i in range(contrast_mean.shape[0])]
        )
    return features, names

def _attack_time(y: np.ndarray, sr: int, cfg: Config) -> float:
    rms = librosa.feature.rms(y=y, frame_length=cfg.frame_length, hop_length=cfg.hop_length)[0]
    if np.all(rms == 0):
        return 0.0
    peak_idx = int(np.argmax(rms))
    threshold = 0.1 * rms[peak_idx]
    onset_idx = int(np.argmax(rms >= threshold))
    times = librosa.frames_to_time(np.arange(len(rms)), sr=sr, hop_length=cfg.hop_length)
    return float(max(0.0, times[peak_idx] - times[onset_idx]))

def extract_pain_markers(y: np.ndarray, sr: int, cfg: Config) -> Tuple[np.ndarray, List[str]]:
    sound = parselmouth.Sound(y, sampling_frequency=sr)
    pitch = sound.to_pitch()
    f0_values = pitch.selected_array["frequency"]
    f0_values = f0_values[f0_values > 0]
    f0_mean = _safe_float(np.mean(f0_values) if f0_values.size else 0.0)
    f0_std = _safe_float(np.std(f0_values) if f0_values.size else 0.0)

    point_process = praat.call(sound, "To PointProcess (periodic, cc)", 75, 500)
    jitter = _safe_float(praat.call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3))
    shimmer = _safe_float(praat.call([sound, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6))

    harmonicity = sound.to_harmonicity_cc()
    hnr = _safe_float(praat.call(harmonicity, "Get mean", 0, 0))

    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr).mean()
    rms = librosa.feature.rms(y=y).mean()
    attack_time = _attack_time(y, sr, cfg)

    features = np.array([
        f0_mean, f0_std, jitter, shimmer, hnr,
        spectral_centroid, rms, attack_time
        ], dtype=float)
    names = [
        "f0_mean", "f0_std", "jitter_local", "shimmer_local", "hnr",
        "spectral_centroid", "rms_energy", "attack_time"
        ]
    return features, names

def extract_combined_features(y: np.ndarray, sr: int, cfg: Config) -> Tuple[np.ndarray, List[str]]:
    mfcc_feats, mfcc_names = extract_mfcc_features(y, sr, cfg)
    pain_feats, pain_names = extract_pain_markers(y, sr, cfg)
    features = np.concatenate([mfcc_feats, pain_feats])
    names = mfcc_names + pain_names
    return features, names

def normalize_label(label: str) -> str:
    return label.strip().lower().replace(" ", "_").replace("-", "_")

def list_audio_files(root_dir: str, exts: Tuple[str, ...] = (".wav", ".mp3", ".3gp", ".m4a", ".flac")) -> List[Tuple[str, str]]:
    items = []
    for dirpath, _, filenames in os.walk(root_dir):
        label = os.path.basename(dirpath)
        for fname in filenames:
            if fname.lower().endswith(exts):
                items.append((os.path.join(dirpath, fname), label))
    return items

In [34]:
def build_dataset_index(cfg: Config) -> pd.DataFrame:
    items = list_audio_files(cfg.data_root)
    if not items:
        raise FileNotFoundError(f"No audio files found in {cfg.data_root}")
    df = pd.DataFrame(items, columns=["file_path", "label"])
    df["label_norm"] = df["label"].apply(normalize_label)
    df["label"] = df["label_norm"].map(cfg.label_map)
    df = df[df["label"].isin(cfg.target_classes)]
    if df.empty:
        raise ValueError("No samples after label mapping. Check cfg.label_map and folder names.")
    return df[["file_path", "label"]]

def extract_features_for_dataset(df: pd.DataFrame, cfg: Config, mode: str, use_cache: bool = True) -> Tuple[np.ndarray, np.ndarray, List[str], List[str]]:
    cache_path = os.path.join(cfg.cache_dir, f"{mode}_{cfg.label_version}_features.npz")
    if use_cache and os.path.exists(cache_path):
        cached = np.load(cache_path, allow_pickle=True)
        return cached["X"], cached["y"], list(cached["feature_names"]), list(cached["file_paths"])

    X_list = []
    y_list = []
    file_paths = []
    feature_names = None
    for idx, row in df.iterrows():
        file_path = row["file_path"]
        label = row["label"]
        try:
            y_audio, sr = load_audio(file_path, cfg.sample_rate)
            if mode == "mfcc":
                feats, names = extract_mfcc_features(y_audio, sr, cfg)
            elif mode == "pain":
                feats, names = extract_pain_markers(y_audio, sr, cfg)
            elif mode == "combined":
                feats, names = extract_combined_features(y_audio, sr, cfg)
            else:
                raise ValueError(f"Unknown mode: {mode}")
            if feature_names is None:
                feature_names = names
            X_list.append(feats)
            y_list.append(label)
            file_paths.append(file_path)
        except Exception as exc:
            print(f"Skip {file_path}: {exc}")
            continue

    X = np.vstack(X_list)
    y = np.array(y_list)
    feature_names = feature_names or []
    np.savez(cache_path, X=X, y=y, feature_names=feature_names, file_paths=file_paths)
    return X, y, feature_names, file_paths

df_index = build_dataset_index(cfg)
df_index["label"].value_counts()

label
hungry     428
pain       427
no_pain    324
Name: count, dtype: int64

In [35]:
def train_model(X: np.ndarray, y: np.ndarray, cfg: Config) -> Tuple[RandomForestClassifier, np.ndarray, np.ndarray, np.ndarray, np.ndarray, LabelEncoder, StandardScaler]:
    label_encoder = LabelEncoder()
    y_enc = label_encoder.fit_transform(y)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_enc, test_size=cfg.test_size, stratify=y_enc, random_state=cfg.random_state
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    model = RandomForestClassifier(
        n_estimators=300, random_state=cfg.random_state, class_weight="balanced"
    )
    model.fit(X_train_scaled, y_train)
    return model, X_train_scaled, X_test_scaled, y_train, y_test, label_encoder, scaler

def compute_class_weights(y: np.ndarray) -> Dict[int, float]:
    from sklearn.utils.class_weight import compute_class_weight
    classes = np.unique(y)
    weights = compute_class_weight("balanced", classes=classes, y=y)
    return {int(cls): float(w) for cls, w in zip(classes, weights)}

def evaluate_model(model, X_test: np.ndarray, y_test: np.ndarray, label_encoder: LabelEncoder) -> Dict[str, object]:
    y_pred = model.predict(X_test)
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "confusion_matrix": confusion_matrix(y_test, y_pred),
        "classification_report": classification_report(
            y_test, y_pred, target_names=label_encoder.classes_, zero_division=0
            )
        }
    return metrics

def plot_confusion(cm: np.ndarray, labels: np.ndarray, title: str) -> None:
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

def plot_feature_importance(model, feature_names: List[str], title: str, top_n: int = 20) -> None:
    if not hasattr(model, 'feature_importances_'):
        print(f"Model {type(model).__name__} does not have feature_importances_")
        return
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1][:top_n]
    plt.figure(figsize=(8, 6))
    sns.barplot(x=importances[indices], y=[feature_names[i] for i in indices], palette="viridis")
    plt.title(title)
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

def print_top_features(model, feature_names: List[str], top_n: int = 10) -> None:
    if not hasattr(model, 'feature_importances_'):
        print(f"Model {type(model).__name__} does not have feature_importances_")
        return
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1][:top_n]
    print(f"\nTop {top_n} Features:")
    for i, idx in enumerate(indices, 1):
        print(f"{i:2d}. {feature_names[idx]:30s} -> {importances[idx]:.5f}")

In [36]:
print("="*70)
print("STEP 1: LOAD DATA & ANALYZE CLASS DISTRIBUTION")
print("="*70)

X, y, feature_names, _ = extract_features_for_dataset(df_index, cfg, mode="combined", use_cache=True)
print(f"\nShape: X={X.shape}, unique labels={np.unique(y)}")
print("\nClass Distribution:")
print(df_index["label"].value_counts())

label_encoder = LabelEncoder()
y_enc = label_encoder.fit_transform(y)
class_weights = compute_class_weights(y_enc)
print("\nComputed Class Weights:")
for cls_idx, cls_name in enumerate(label_encoder.classes_):
    print(f"  {cls_name}: {class_weights[cls_idx]:.3f}")

print(f"\nFeature count: {len(feature_names)}")
print(f"Training set size: {X.shape[0]}")


STEP 1: LOAD DATA & ANALYZE CLASS DISTRIBUTION

Shape: X=(1179, 131), unique labels=['hungry' 'no_pain' 'pain']

Class Distribution:
label
hungry     428
pain       427
no_pain    324
Name: count, dtype: int64

Computed Class Weights:
  hungry: 0.918
  no_pain: 1.213
  pain: 0.920

Feature count: 131
Training set size: 1179


In [37]:
print("="*70)
print("STEP 2: TRAIN-TEST SPLIT & SCALING")
print("="*70)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=cfg.test_size, stratify=y_enc, random_state=cfg.random_state
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTrain set: {X_train_scaled.shape}")
print(f"Test set:  {X_test_scaled.shape}")
print(f"\nScaler mean range: [{scaler.mean_.min():.3f}, {scaler.mean_.max():.3f}]")
print(f"Scaler scale range: [{scaler.scale_.min():.3f}, {scaler.scale_.max():.3f}]")


STEP 2: TRAIN-TEST SPLIT & SCALING

Train set: (943, 131)
Test set:  (236, 131)

Scaler mean range: [-350.046, 1479.324]
Scaler scale range: [0.012, 479.183]


In [38]:
print("="*70)
print("STEP 3: HYPERPARAMETER TUNING WITH GRIDSEARCHCV")
print("="*70)

rf_params = {
    'n_estimators': [100, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
}

print("\n📊 RandomForest GridSearchCV...")
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=cfg.random_state, class_weight='balanced'),
    rf_params,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
rf_grid.fit(X_train_scaled, y_train)

print(f"\nBest RandomForest params: {rf_grid.best_params_}")
print(f"Best CV F1 score: {rf_grid.best_score_:.4f}")

if HAS_XGBOOST:
    print("\n📊 XGBoost GridSearchCV...")
    xgb_params = {
        'n_estimators': [100, 300],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1],
    }
    xgb_grid = GridSearchCV(
        XGBClassifier(random_state=cfg.random_state, use_label_encoder=False, eval_metric='mlogloss'),
        xgb_params,
        cv=5,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=1
    )
    xgb_grid.fit(X_train_scaled, y_train)
    print(f"\nBest XGBoost params: {xgb_grid.best_params_}")
    print(f"Best CV F1 score: {xgb_grid.best_score_:.4f}")
else:
    xgb_grid = None


STEP 3: HYPERPARAMETER TUNING WITH GRIDSEARCHCV

📊 RandomForest GridSearchCV...
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Best RandomForest params: {'max_depth': 30, 'min_samples_split': 2, 'n_estimators': 100}
Best CV F1 score: 0.3579


In [39]:
print("="*70)
print("STEP 4: FEATURE SELECTION (SelectFromModel)")
print("="*70)

print(f"\nOriginal feature count: {X_train_scaled.shape[1]}")

selector = SelectFromModel(rf_grid.best_estimator_, prefit=True, threshold='median')
X_train_selected = selector.transform(X_train_scaled)
X_test_selected = selector.transform(X_test_scaled)
selected_features = [feature_names[i] for i in range(len(feature_names)) if selector.get_support()[i]]

print(f"Selected feature count: {X_train_selected.shape[1]}")
print(f"\nSelected features ({len(selected_features)}):")
for i, fname in enumerate(selected_features, 1):
    print(f"  {i:2d}. {fname}")


STEP 4: FEATURE SELECTION (SelectFromModel)

Original feature count: 131
Selected feature count: 66

Selected features (66):
   1. mfcc_mean_0
   2. mfcc_mean_1
   3. mfcc_mean_2
   4. mfcc_mean_3
   5. mfcc_mean_5
   6. mfcc_mean_7
   7. mfcc_mean_8
   8. mfcc_mean_9
   9. mfcc_mean_10
  10. mfcc_mean_11
  11. mfcc_mean_12
  12. mfcc_mean_13
  13. mfcc_mean_16
  14. mfcc_mean_17
  15. mfcc_mean_18
  16. mfcc_std_0
  17. mfcc_std_1
  18. mfcc_std_2
  19. mfcc_std_3
  20. mfcc_std_4
  21. mfcc_std_5
  22. mfcc_std_6
  23. mfcc_std_7
  24. mfcc_std_8
  25. mfcc_std_9
  26. mfcc_std_10
  27. mfcc_std_11
  28. mfcc_std_12
  29. mfcc_std_13
  30. mfcc_std_14
  31. mfcc_std_15
  32. mfcc_std_16
  33. mfcc_std_17
  34. mfcc_std_19
  35. mel_mean_1
  36. mel_mean_3
  37. mel_mean_4
  38. mel_mean_6
  39. mel_mean_7
  40. mel_mean_14
  41. mel_mean_45
  42. mel_mean_46
  43. mel_mean_47
  44. chroma_mean_0
  45. chroma_mean_2
  46. chroma_mean_3
  47. chroma_mean_4
  48. chroma_mean_5
  49. chr

In [40]:
print("="*70)
print("STEP 5: TRAIN ALL MODELS (FULL FEATURES & SELECTED)")
print("="*70)

tuned_models = {
    'RandomForest (Tuned)': rf_grid.best_estimator_,
    'GradientBoosting (Default)': GradientBoostingClassifier(random_state=cfg.random_state, max_depth=5, learning_rate=0.1, n_estimators=300),
}

if HAS_XGBOOST and xgb_grid is not None:
    tuned_models['XGBoost (Tuned)'] = xgb_grid.best_estimator_

results_full = {}
results_selected = {}

for name, model in tuned_models.items():
    print(f"\n📈 {name}")
    print(" "*50)
    
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision_macro': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_test, y_pred, average='macro', zero_division=0),
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'classification_report': classification_report(y_test, y_pred, target_names=label_encoder.classes_, zero_division=0),
        'y_pred': y_pred,
    }
    results_full[name] = metrics
    
    print(f"Accuracy:  {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision_macro']:.4f}")
    print(f"Recall:    {metrics['recall_macro']:.4f}")
    print(f"F1 Macro:  {metrics['f1_macro']:.4f}")

    if hasattr(model, 'feature_importances_'):
        print_top_features(model, feature_names, top_n=10)

print("\n" + "="*70)
print("TRAINING ON SELECTED FEATURES ONLY")
print("="*70)

for name, model in tuned_models.items():
    print(f"\n📈 {name} (Selected Features)")
    print(" "*50)
    
    model_clone = type(model)(**model.get_params())
    model_clone.fit(X_train_selected, y_train)
    y_pred = model_clone.predict(X_test_selected)
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision_macro': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_test, y_pred, average='macro', zero_division=0),
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'classification_report': classification_report(y_test, y_pred, target_names=label_encoder.classes_, zero_division=0),
        'y_pred': y_pred,
        'model': model_clone,
    }
    results_selected[name] = metrics
    
    print(f"Accuracy:  {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision_macro']:.4f}")
    print(f"Recall:    {metrics['recall_macro']:.4f}")
    print(f"F1 Macro:  {metrics['f1_macro']:.4f}")


STEP 5: TRAIN ALL MODELS (FULL FEATURES & SELECTED)

📈 RandomForest (Tuned)
                                                  
Accuracy:  0.3263
Precision: 0.3413
Recall:    0.3251
F1 Macro:  0.3311

Top 10 Features:
 1. mfcc_std_15                    -> 0.01684
 2. mfcc_std_1                     -> 0.01623
 3. mfcc_std_14                    -> 0.01508
 4. mfcc_mean_11                   -> 0.01420
 5. contrast_mean_5                -> 0.01295
 6. mfcc_std_12                    -> 0.01176
 7. mfcc_mean_3                    -> 0.01157
 8. mfcc_mean_2                    -> 0.01157
 9. contrast_mean_6                -> 0.01148
10. contrast_mean_3                -> 0.01101

📈 GradientBoosting (Default)
                                                  
Accuracy:  0.3432
Precision: 0.3588
Recall:    0.3406
F1 Macro:  0.3470

Top 10 Features:
 1. contrast_mean_5                -> 0.03549
 2. mfcc_std_14                    -> 0.03338
 3. mfcc_std_1                     -> 0.03076
 4. mfcc_std_1

In [41]:
print("="*70)
print("STEP 6: EVALUATION SUMMARY & COMPARISON TABLE")
print("="*70)

comparison_rows = []
print("\n📋 FULL FEATURES:")
print("-"*70)
for name, metrics in results_full.items():
    comparison_rows.append({
        'Model': name,
        'Features': 'Full',
        'Accuracy': metrics['accuracy'],
        'Precision': metrics['precision_macro'],
        'Recall': metrics['recall_macro'],
        'F1-Score': metrics['f1_macro'],
    })
    print(f"{name:30s} | Acc={metrics['accuracy']:.4f} | F1={metrics['f1_macro']:.4f}")

print("\n📋 SELECTED FEATURES:")
print("-"*70)
for name, metrics in results_selected.items():
    comparison_rows.append({
        'Model': name,
        'Features': 'Selected',
        'Accuracy': metrics['accuracy'],
        'Precision': metrics['precision_macro'],
        'Recall': metrics['recall_macro'],
        'F1-Score': metrics['f1_macro'],
    })
    print(f"{name:30s} | Acc={metrics['accuracy']:.4f} | F1={metrics['f1_macro']:.4f}")

comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values('F1-Score', ascending=False)
print("\n" + "="*70)
print("FINAL COMPARISON TABLE")
print("="*70)
print(comparison_df.to_string(index=False))


STEP 6: EVALUATION SUMMARY & COMPARISON TABLE

📋 FULL FEATURES:
----------------------------------------------------------------------
RandomForest (Tuned)           | Acc=0.3263 | F1=0.3311
GradientBoosting (Default)     | Acc=0.3432 | F1=0.3470

📋 SELECTED FEATURES:
----------------------------------------------------------------------
RandomForest (Tuned)           | Acc=0.3305 | F1=0.3355
GradientBoosting (Default)     | Acc=0.3390 | F1=0.3432

FINAL COMPARISON TABLE
                     Model Features  Accuracy  Precision   Recall  F1-Score
GradientBoosting (Default)     Full  0.343220   0.358818 0.340640  0.346983
GradientBoosting (Default) Selected  0.338983   0.355791 0.336673  0.343239
      RandomForest (Tuned) Selected  0.330508   0.343775 0.330264  0.335498
      RandomForest (Tuned)     Full  0.326271   0.341307 0.325090  0.331050


In [42]:
print("="*70)
print("STEP 7: DIAGNOSTIC ANALYSIS")
print("="*70)

best_result_name = comparison_df.iloc[0]['Model']
best_result_features = comparison_df.iloc[0]['Features']

if best_result_features == 'Full':
    best_metrics = results_full[best_result_name]
    best_X_test = X_test_scaled
else:
    best_metrics = results_selected[best_result_name]
    best_X_test = X_test_selected

print(f"\n🌟 Best Model: {best_result_name}")
print(f"   Features: {best_result_features}")
print(f"   F1 Score: {best_metrics['f1_macro']:.4f}")

cm = best_metrics['confusion_matrix']
print(f"\nConfusion Matrix:")
print(cm)

print(f"\n🔍 Per-Class Recall (diagonal in CM):")
for i, cls_name in enumerate(label_encoder.classes_):
    recall = cm[i, i] / cm[i].sum() if cm[i].sum() > 0 else 0
    print(f"  {cls_name:15s}: {recall:.4f}")

print(f"\n🔍 Per-Class Precision:")
for j, cls_name in enumerate(label_encoder.classes_):
    precision = cm[j, j] / cm[:, j].sum() if cm[:, j].sum() > 0 else 0
    print(f"  {cls_name:15s}: {precision:.4f}")

print(f"\n{best_metrics['classification_report']}")

if hasattr(results_full[list(results_full.keys())[0]].get('model'), 'feature_importances_') or hasattr(results_full[list(results_full.keys())[0]].get('y_pred'), '__class__'):
    model_for_importance = tuned_models[list(tuned_models.keys())[0]]
    if hasattr(model_for_importance, 'feature_importances_'):
        print(f"\n📋 Top 15 Features (from {list(tuned_models.keys())[0]}):")
        print_top_features(model_for_importance, feature_names, top_n=15)

print("\n📚 WHY PAIN vs HUNGRY CONFUSION:")
print("-"*70)
print("Possible reasons:")
print("  1. Both involve intense vocalization → similar acoustic features")
print("  2. Spectral peaks & dynamics may overlap")
print("  3. F0 range of pain & hungry cries might be similar")
print("  4. Limited training data → hard to learn boundaries")
print("  5. Need domain-expert labeling or more pain-specific markers")


STEP 7: DIAGNOSTIC ANALYSIS

🌟 Best Model: GradientBoosting (Default)
   Features: Full
   F1 Score: 0.3470

Confusion Matrix:
[[25 24 37]
 [35 20 10]
 [43  6 36]]

🔍 Per-Class Recall (diagonal in CM):
  hungry         : 0.2907
  no_pain        : 0.3077
  pain           : 0.4235

🔍 Per-Class Precision:
  hungry         : 0.2427
  no_pain        : 0.4000
  pain           : 0.4337

              precision    recall  f1-score   support

      hungry       0.24      0.29      0.26        86
     no_pain       0.40      0.31      0.35        65
        pain       0.43      0.42      0.43        85

    accuracy                           0.34       236
   macro avg       0.36      0.34      0.35       236
weighted avg       0.35      0.34      0.35       236


📋 Top 15 Features (from RandomForest (Tuned)):

Top 15 Features:
 1. mfcc_std_15                    -> 0.01684
 2. mfcc_std_1                     -> 0.01623
 3. mfcc_std_14                    -> 0.01508
 4. mfcc_mean_11                

## Next Steps to Improve Further

1. **Data Augmentation**: Synthetic cry samples via pitch-shift, time-stretch
2. **Domain Features**: Add temporal features (cry duration, pause patterns)
3. **Ensemble Methods**: Stacking or voting with tuned models
4. **Class Balancing**: Weighted sampling or oversampling (SMOTE)
5. **Data Collection**: More pain samples (often underrepresented)
6. **Regularization**: L1/L2 penalties, dropout (if using neural networks)
7. **Expert Review**: Validate confusing pain vs hungry misclassifications